# Excel Reporting & Data Analysis Automation (Supply Chain)

In [7]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

# ================= CONFIG =================
INPUT_FILE = r"C:\Users\Abubakar\OneDrive\Desktop\vendor_data.xlsx"
OUTPUT_FILE = r"C:\Users\Abubakar\OneDrive\Desktop\Abu_Report.xlsx"

# Create folder if not exists
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# ================= LOAD DATA =================
df = pd.read_excel(INPUT_FILE)

# ================= CLEANING =================
df.columns = df.columns.str.strip()
df.dropna(subset=['Email'], inplace=True)
df['Due_Date'] = pd.to_datetime(df['Due_Date'], errors='coerce')

# ================= FEATURE ENGINEERING =================
df['Total_Amount'] = df['Quantity'] * df['Amount']
today = pd.Timestamp.today()

df['Days_Overdue'] = (today - df['Due_Date']).dt.days
df['Days_Overdue'] = df['Days_Overdue'].apply(lambda x: x if x > 0 else 0)

# ================= SEGMENT =================
pending_df = df[df['Payment_Status'] == 'Pending']
overdue_df = df[df['Payment_Status'] == 'Overdue']
paid_df = df[df['Payment_Status'] == 'Paid']

# ================= PIVOTS =================
pivot_status = pd.pivot_table(df, values='Total_Amount', index='Payment_Status', aggfunc='sum')
pivot_city = pd.pivot_table(df, values='Total_Amount', index='City', aggfunc='sum')

# ================= SUMMARY =================
summary = pd.DataFrame({
    "Metric": ["Total Vendors", "Total Amount", "Pending", "Overdue", "Paid"],
    "Value": [
        len(df),
        df['Total_Amount'].sum(),
        pending_df['Total_Amount'].sum(),
        overdue_df['Total_Amount'].sum(),
        paid_df['Total_Amount'].sum()
    ]
})

# ================= WRITE EXCEL =================
with pd.ExcelWriter(OUTPUT_FILE, engine='xlsxwriter') as writer:

    workbook = writer.book

    # ===== FORMATS =====
    title_format = workbook.add_format({
        'bold': True, 'font_size': 18, 'align': 'center'
    })

    header_format = workbook.add_format({
        'bold': True,
        'bg_color': '#2F75B5',
        'color': 'white',
        'border': 1,
        'align': 'center'
    })

    money_format = workbook.add_format({'num_format': '₹#,##0'})
    overdue_format = workbook.add_format({'bg_color': '#FFC7CE'})

    # ================= COVER PAGE =================
    cover = workbook.add_worksheet('Report Overview')
    cover.merge_range('B2:H3', 'Vendor Payment Report', title_format)
    cover.write('B5', f"Generated on: {datetime.now().strftime('%d-%m-%Y %H:%M')}")

    # ================= FUNCTION FOR FORMATTED SHEET =================
    def format_sheet(data, sheet_name):
        data.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]

        # Header styling
        for col_num, col_name in enumerate(data.columns):
            ws.write(0, col_num, col_name, header_format)
            ws.set_column(col_num, col_num, 20)

        # Freeze top row
        ws.freeze_panes(1, 0)

        # Add filter
        ws.autofilter(0, 0, len(data), len(data.columns)-1)

        # Currency format
        if 'Total_Amount' in data.columns:
            col_idx = data.columns.get_loc('Total_Amount')
            ws.set_column(col_idx, col_idx, 18, money_format)

    # ================= WRITE DATA =================
    format_sheet(df, 'Full Data')
    format_sheet(pending_df, 'Pending')
    format_sheet(overdue_df, 'Overdue')
    format_sheet(paid_df, 'Paid')

    pivot_status.to_excel(writer, sheet_name='Pivot_Status')
    pivot_city.to_excel(writer, sheet_name='Pivot_City')
    summary.to_excel(writer, sheet_name='Summary', index=False)

    # ================= DASHBOARD =================
    dashboard = workbook.add_worksheet('Dashboard')

    dashboard.write('B1', '📊 Dashboard Overview', title_format)

    # BAR CHART
    chart1 = workbook.add_chart({'type': 'column'})
    chart1.add_series({
        'name': 'Payment Status',
        'categories': '=Pivot_Status!A2:A4',
        'values': '=Pivot_Status!B2:B4',
        'data_labels': {'value': True},
    })
    chart1.set_title({'name': 'Payment Status'})
    dashboard.insert_chart('B3', chart1)

    # PIE CHART
    chart2 = workbook.add_chart({'type': 'pie'})
    chart2.add_series({
        'categories': '=Pivot_Status!A2:A4',
        'values': '=Pivot_Status!B2:B4',
        'data_labels': {'percentage': True},
    })
    chart2.set_title({'name': 'Payment Share'})
    dashboard.insert_chart('B20', chart2)

    # CITY CHART
    chart3 = workbook.add_chart({'type': 'bar'})
    chart3.add_series({
        'categories': '=Pivot_City!A2:A10',
        'values': '=Pivot_City!B2:B10',
        'data_labels': {'value': True},
    })
    chart3.set_title({'name': 'City Wise'})
    dashboard.insert_chart('J3', chart3)

# ================= AUTO OPEN =================
os.startfile(OUTPUT_FILE)

print("Professional Abu_Report Generated!")
print("Location:", OUTPUT_FILE)

🔥 Professional Abu_Report Generated!
📂 Location: C:\Users\Abubakar\OneDrive\Desktop\Abu_Report.xlsx
